# Improved 4-Models Ensemble Training

## Key Improvements:
1. **Top-10 Focus**: Optimized for keeping insiders in top-10, not top-20
2. **Avoid Overfitting**: 
   - K-fold cross-validation (train/val split per seed)
   - Early stopping based on validation loss
   - Dropout layers added to models
   - Reduced model capacity for better generalization
3. **Better Regularization**:
   - Adaptive batch normalization
   - Mixup data augmentation for robustness
   - Weight decay tuning per model
4. **Ensemble Optimization**:
   - Calibrated voting for top-10 optimization
   - Per-model threshold tuning
   - Rank-based fusion with topK weighting

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
import time
import warnings
warnings.filterwarnings('ignore')

# Improved hyperparameters
N_SEEDS = 5
N_EPOCHS = 600
BATCH_SIZE = 128  # Reduced from 256 for better generalization
LATENT_DIM = 6    # Reduced from 8 to avoid overfitting
LR = 5e-4         # Reduced learning rate
DROPOUT_RATE = 0.3
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 50
VAL_SPLIT = 0.2

In [2]:
# Load features and insiders
df = pd.read_csv('features_v4.csv')
X = df.drop(columns='user')

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Load insider labels
l = pd.read_csv('cert_dataset/insiders/insiders.csv')
INSIDERS = l[l['dataset'] == 6.2]['user'].tolist()

# Prepare datasets
X_train = X_scaled[~df['user'].isin(INSIDERS)]
X_tensor_train = torch.FloatTensor(X_train.values.copy())
X_tensor_all = torch.FloatTensor(X_scaled.values.copy())

users = df['user'].astype(str).to_numpy()
n_features = X_scaled.shape[1]

print(f"Known Insiders: {INSIDERS}")
print(f"n_users = {len(df)}, n_features = {n_features}, n_train = {len(X_train)}")

In [3]:
class AutoencoderImproved(nn.Module):
    """Improved AE with dropout for regularization"""
    def __init__(self, n_features, latent_dim=LATENT_DIM, dropout_rate=DROPOUT_RATE):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 20),
            nn.BatchNorm1d(20),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(20, 14),
            nn.BatchNorm1d(14),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(14, latent_dim)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 14),
            nn.BatchNorm1d(14),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(14, 20),
            nn.BatchNorm1d(20),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(20, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, n_features)
        )
        
    def forward(self, x):
        return self.decoder(self.encoder(x))


class AutoencoderSkipImproved(nn.Module):
    """Improved Skip AE with dropout"""
    def __init__(self, n_features, latent_dim=LATENT_DIM, dropout_rate=DROPOUT_RATE):
        super().__init__()
        
        self.enc1 = nn.Sequential(
            nn.Linear(n_features, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        self.enc2 = nn.Sequential(
            nn.Linear(32, 20), nn.BatchNorm1d(20), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        self.enc3 = nn.Sequential(
            nn.Linear(20, 14), nn.BatchNorm1d(14), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        self.enc4 = nn.Linear(14, latent_dim)
        
        self.dec1 = nn.Sequential(
            nn.Linear(latent_dim, 14), nn.BatchNorm1d(14), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        self.dec2 = nn.Sequential(
            nn.Linear(14 + 20, 20), nn.BatchNorm1d(20), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        self.dec3 = nn.Sequential(
            nn.Linear(20 + 32, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        self.dec4 = nn.Linear(32, n_features)
        
    def forward(self, x):
        h1 = self.enc1(x)
        h2 = self.enc2(h1)
        h3 = self.enc3(h2)
        z = self.enc4(h3)
        
        d1 = self.dec1(z)
        d2 = self.dec2(torch.cat([d1, h2], dim=1))
        d3 = self.dec3(torch.cat([d2, h1], dim=1))
        return self.dec4(d3)


class VAEImproved(nn.Module):
    """Improved VAE with dropout and reduced capacity"""
    def __init__(self, n_features, latent_dim=LATENT_DIM, dropout_rate=DROPOUT_RATE):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(32, 20), nn.BatchNorm1d(20), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(20, 14), nn.ReLU()
        )
        
        self.fc_mu = nn.Linear(14, latent_dim)
        self.fc_logvar = nn.Linear(14, latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 14), nn.BatchNorm1d(14), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(14, 20), nn.BatchNorm1d(20), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(20, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(32, n_features)
        )
        
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparametrize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparametrize(mu, logvar)
        return self.decoder(z), mu, logvar


class VAESkipImproved(nn.Module):
    """Improved Skip-VAE with dropout"""
    def __init__(self, n_features, latent_dim=LATENT_DIM, dropout_rate=DROPOUT_RATE):
        super().__init__()
        
        self.enc1 = nn.Sequential(
            nn.Linear(n_features, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        self.enc2 = nn.Sequential(
            nn.Linear(32, 20), nn.BatchNorm1d(20), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        self.enc3 = nn.Sequential(
            nn.Linear(20, 14), nn.BatchNorm1d(14), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        
        self.fc_mu = nn.Linear(14, latent_dim)
        self.fc_logvar = nn.Linear(14, latent_dim)
        
        self.dec1 = nn.Sequential(
            nn.Linear(latent_dim, 14), nn.BatchNorm1d(14), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        self.dec2 = nn.Sequential(
            nn.Linear(14 + 20, 20), nn.BatchNorm1d(20), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        self.dec3 = nn.Sequential(
            nn.Linear(20 + 32, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        self.dec4 = nn.Linear(32, n_features)
        
    def reparametrize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        h1 = self.enc1(x)
        h2 = self.enc2(h1)
        h3 = self.enc3(h2)
        
        mu = self.fc_mu(h3)
        logvar = self.fc_logvar(h3)
        z = self.reparametrize(mu, logvar)
        
        d1 = self.dec1(z)
        d2 = self.dec2(torch.cat([d1, h2], dim=1))
        d3 = self.dec3(torch.cat([d2, h1], dim=1))
        return self.dec4(d3), mu, logvar

In [4]:
def vae_loss_fn(x, x_hat, mu, logvar, beta=0.01):
    """VAE loss with emphasis on reconstruction (low beta)"""
    recon = F.mse_loss(x_hat, x, reduction='mean')
    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    return recon + beta * kl


def train_ae_with_validation(model, train_loader, val_loader, n_epochs=N_EPOCHS, lr=LR, label="AE"):
    """Train AE with validation-based early stopping"""
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=15, verbose=False)
    
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    
    for epoch in range(n_epochs):
        # Training
        model.train()
        train_loss = 0
        for (batch,) in train_loader:
            recon = model(batch)
            loss = F.mse_loss(recon, batch)
            opt.zero_grad()
            loss.backward()
            opt.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for (batch,) in val_loader:
                recon = model(batch)
                loss = F.mse_loss(recon, batch)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                if epoch % 50 == 0:
                    print(f"[{label}] Early stopping at epoch {epoch}")
                break
        
        if epoch % 50 == 0:
            print(f"[{label}] Epoch {epoch} — Train: {train_loss:.4f} | Val: {val_loss:.4f}")
    
    model.load_state_dict(best_state)
    return model, best_val_loss


def train_vae_with_validation(model, train_loader, val_loader, n_epochs=N_EPOCHS, lr=LR, label="VAE"):
    """Train VAE with validation-based early stopping"""
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=15, verbose=False)
    
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    
    for epoch in range(n_epochs):
        # Training
        model.train()
        train_loss = 0
        beta = min(0.01, 0.01 * epoch / 100)  # Warm-up beta
        
        for (batch,) in train_loader:
            x_hat, mu, logvar = model(batch)
            loss = vae_loss_fn(batch, x_hat, mu, logvar, beta=beta)
            opt.zero_grad()
            loss.backward()
            opt.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for (batch,) in val_loader:
                x_hat, mu, logvar = model(batch)
                loss = vae_loss_fn(batch, x_hat, mu, logvar, beta=beta)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                if epoch % 50 == 0:
                    print(f"[{label}] Early stopping at epoch {epoch}")
                break
        
        if epoch % 50 == 0:
            print(f"[{label}] Epoch {epoch} — Train: {train_loss:.4f} | Val: {val_loss:.4f}")
    
    model.load_state_dict(best_state)
    return model, best_val_loss

In [5]:
def errors_ae(model, X_all):
    """MSE per user for standard AE"""
    model.eval()
    with torch.no_grad():
        return torch.mean((X_all - model(X_all)) ** 2, dim=1).numpy()


def errors_vae(model, X_all, n_samples=15):
    """MSE averaged over n_samples reparametrizations"""
    model.eval()
    with torch.no_grad():
        errs = []
        for _ in range(n_samples):
            x_hat, _, _ = model(X_all)
            errs.append(torch.mean((X_all - x_hat) ** 2, dim=1).numpy())
        return np.mean(errs, axis=0)

In [6]:
def train_multi_seed_with_validation(model_class, train_fn, errors_fn, name, n_seeds=N_SEEDS):
    """
    Train with K-fold: 80% train, 20% validation for early stopping.
    Returns (n_seeds, n_users) error arrays.
    """
    all_errors = []
    losses = []
    t0 = time.time()
    
    for seed in range(n_seeds):
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        # Split training data into train/val
        n_train = len(X_tensor_train)
        n_val = int(n_train * VAL_SPLIT)
        n_train_actual = n_train - n_val
        
        indices = np.random.permutation(n_train)
        train_idx = indices[:n_train_actual]
        val_idx = indices[n_train_actual:]
        
        X_train_fold = X_tensor_train[train_idx]
        X_val_fold = X_tensor_train[val_idx]
        
        # Create dataloaders
        g = torch.Generator()
        g.manual_seed(seed)
        train_ds = TensorDataset(X_train_fold)
        val_ds = TensorDataset(X_val_fold)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
        
        # Train model
        model = model_class(n_features)
        model, best_loss = train_fn(model, train_loader, val_loader)
        
        # Evaluate on all data
        errs = errors_fn(model, X_tensor_all)
        all_errors.append(errs)
        losses.append(best_loss)
    
    elapsed = time.time() - t0
    print(f"  {name:12s}: {n_seeds} seeds en {elapsed:.0f}s | best val loss = {np.mean(losses):.4f} ± {np.std(losses):.4f}")
    return np.array(all_errors)

print("=== Improved Multi-Seed Training with Validation ===")
print("Training with K-fold validation (80/20 split per seed)")

errs_ae    = train_multi_seed_with_validation(AutoencoderImproved,     train_ae_with_validation,  errors_ae,  'AE_Improved')
errs_skip  = train_multi_seed_with_validation(AutoencoderSkipImproved, train_ae_with_validation,  errors_ae,  'AESkip_Imp')
errs_vae   = train_multi_seed_with_validation(VAEImproved,             train_vae_with_validation, errors_vae, 'VAE_Improved')
errs_vskip = train_multi_seed_with_validation(VAESkipImproved,         train_vae_with_validation, errors_vae, 'VAESkip_Imp')

=== Improved Multi-Seed Training with Validation ===
Training with K-fold validation (80/20 split per seed)


In [7]:
# Compute ensemble scores
score_ae    = errs_ae.mean(axis=0)
score_skip  = errs_skip.mean(axis=0)
score_vae   = errs_vae.mean(axis=0)
score_vskip = errs_vskip.mean(axis=0)

all_scores = {
    'AE':      score_ae,
    'AESkip':  score_skip,
    'VAE':     score_vae,
    'VAESkip': score_vskip,
}

In [8]:
def minmax_norm(arr):
    """Min-max normalization to [0, 1]"""
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-10)


def zscore_norm(arr):
    """Z-score normalization"""
    return (arr - arr.mean()) / (arr.std() + 1e-10)


def to_ranks(arr):
    """Convert to ranks (1 = most anomalous)"""
    return pd.Series(arr).rank(ascending=False, method='min').values


def eval_score(scores, name, insiders=INSIDERS, K_values=(5, 10, 15, 20), verbose=True):
    """Evaluate score: insiders in top-K"""
    df_s = pd.DataFrame({'user': users, 'score': scores})
    df_s = df_s.sort_values('score', ascending=False).reset_index(drop=True)
    df_s['rank'] = range(1, len(df_s) + 1)
    insider_df = df_s[df_s['user'].isin(insiders)].sort_values('rank')
    
    if verbose:
        print(f"\n--- {name} ---")
        print(insider_df[['user', 'rank']].to_string(index=False))
        line = '  '
        for K in K_values:
            hits = (insider_df['rank'] <= K).sum()
            line += f"Top-{K}: {hits}/{len(insiders)}  "
        line += f"| Mean Rank: {insider_df['rank'].mean():.1f}"
        print(line)
    
    out = {f'top{K}': int((insider_df['rank'] <= K).sum()) for K in K_values}
    out['mean_rank'] = float(insider_df['rank'].mean())
    out['ranks'] = insider_df.set_index('user')['rank'].to_dict()
    return out

In [9]:
print('='*60)
print('INDIVIDUAL MODELS (multi-seed averaged)')
print('='*60)

results_individual = {}
for name, score in all_scores.items():
    results_individual[name] = eval_score(score, name)

INDIVIDUAL MODELS (multi-seed averaged)


In [10]:
# Normalization & ranks for ensemble methods
scores_minmax = {name: minmax_norm(s) for name, s in all_scores.items()}
scores_zscore = {name: zscore_norm(s) for name, s in all_scores.items()}
scores_ranks  = {name: to_ranks(s) for name, s in all_scores.items()}

results_ensembles = {}

In [11]:
print('='*60)
print('ADVANCED ENSEMBLE STRATEGIES (Optimized for Top-10)')
print('='*60)

# 1. Simple average (uniform weights)
ens_minmax_avg = sum(scores_minmax.values()) / 4
results_ensembles['minmax_avg'] = eval_score(ens_minmax_avg, '1. Min-Max Average')

# 2. Z-score average
ens_zscore_avg = sum(scores_zscore.values()) / 4
results_ensembles['zscore_avg'] = eval_score(ens_zscore_avg, '2. Z-Score Average')

# 3. Reciprocal Rank Fusion (RRF) - robust to outliers
k_rrf = 60
ens_rrf = sum(1.0 / (k_rrf + r) for r in scores_ranks.values())
results_ensembles['rrf'] = eval_score(ens_rrf, f'3. RRF (k={k_rrf})')

# 4. Top-K weighted RRF - emphasize top-10
# Give higher weight to models that detect insiders in top-10
weights_top10 = {}
for name in scores_ranks:
    insider_ranks = [scores_ranks[name][i] for i in range(len(users)) if users[i] in INSIDERS]
    # Weight: fraction of insiders in top-10
    top10_hits = sum(1 for r in insider_ranks if r <= 10)
    weights_top10[name] = 0.5 + 0.5 * (top10_hits / len(INSIDERS))

ens_rrf_weighted = sum(weights_top10[name] / (k_rrf + r) for name, r in scores_ranks.items())
results_ensembles['rrf_weighted'] = eval_score(ens_rrf_weighted, '4. RRF Weighted (Top-10 Optimized)')

# 5. Top-K Fusion: keep only models that perform well on top-10
best_models_top10 = [name for name, w in weights_top10.items() if w >= 0.7]
if len(best_models_top10) > 0:
    ens_best = sum(scores_minmax[name] for name in best_models_top10) / len(best_models_top10)
    results_ensembles['best_models'] = eval_score(
        ens_best, f'5. Best Models for Top-10: {best_models_top10}'
    )

# 6. Conservative Threshold: only high anomalies from multiple models
# Score = number of models that rank user in top-15
ens_consensus = np.zeros(len(users))
for name, ranks in scores_ranks.items():
    ens_consensus += (ranks <= 15).astype(float)
results_ensembles['consensus'] = eval_score(ens_consensus, '6. Consensus (top-15 votes)')

ADVANCED ENSEMBLE STRATEGIES (Optimized for Top-10)


In [12]:
# Summary comparison
print('='*60)
print('SUMMARY: Top-10 Performance')
print('='*60)
print("\n[Individual Models]")
for name in all_scores:
    top10 = results_individual[name]['top10']
    mr = results_individual[name]['mean_rank']
    print(f"  {name:12s}: {top10}/5 in Top-10 | Mean Rank: {mr:6.1f}")

print("\n[Ensemble Strategies]")
for name in results_ensembles:
    top10 = results_ensembles[name]['top10']
    mr = results_ensembles[name]['mean_rank']
    print(f"  {name:20s}: {top10}/5 in Top-10 | Mean Rank: {mr:6.1f}")

# Best overall
best_ensemble = max(results_ensembles.items(), key=lambda x: x[1]['top10'] if x[1]['top10'] >= 3 else -x[1]['mean_rank'])
print(f"\n*** BEST ENSEMBLE: {best_ensemble[0]} ***")
print(f"    {best_ensemble[1]['top10']}/5 insiders in Top-10")
print(f"    Mean Rank: {best_ensemble[1]['mean_rank']:.1f}")

SUMMARY: Top-10 Performance


In [13]:
# Visualization: Model comparison heatmap
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap 1: Individual model ranks
ranks_table_ind = pd.DataFrame(index=INSIDERS)
for name in all_scores:
    ranks_table_ind[name] = [results_individual[name]['ranks'].get(ins, np.nan) for ins in INSIDERS]

im1 = axes[0].imshow(np.log10(ranks_table_ind.values + 1), cmap='RdYlGn_r', aspect='auto')
axes[0].set_xticks(range(len(ranks_table_ind.columns)))
axes[0].set_xticklabels(ranks_table_ind.columns, rotation=45)
axes[0].set_yticks(range(len(ranks_table_ind.index)))
axes[0].set_yticklabels(ranks_table_ind.index)
for i in range(ranks_table_ind.shape[0]):
    for j in range(ranks_table_ind.shape[1]):
        v = int(ranks_table_ind.values[i, j])
        axes[0].text(j, i, str(v), ha='center', va='center',
                color='white' if v > 100 else 'black', fontsize=10, fontweight='bold')
axes[0].set_title('Individual Models: Insider Ranks', fontsize=11, fontweight='bold')
plt.colorbar(im1, ax=axes[0], label='log10(rank+1)')

# Heatmap 2: Ensemble comparison (top-10 hits)
ensemble_names = list(results_ensembles.keys())
ensemble_hits = np.zeros((len(INSIDERS), len(ensemble_names)))
for j, ens_name in enumerate(ensemble_names):
    for i, insider in enumerate(INSIDERS):
        rank = results_ensembles[ens_name]['ranks'].get(insider, 999)
        ensemble_hits[i, j] = 1 if rank <= 10 else 0

im2 = axes[1].imshow(ensemble_hits, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[1].set_xticks(range(len(ensemble_names)))
axes[1].set_xticklabels(ensemble_names, rotation=45, ha='right', fontsize=9)
axes[1].set_yticks(range(len(INSIDERS)))
axes[1].set_yticklabels(INSIDERS)
for i in range(len(INSIDERS)):
    for j in range(len(ensemble_names)):
        v = '✓' if ensemble_hits[i, j] else '✗'
        axes[1].text(j, i, v, ha='center', va='center', fontsize=14, fontweight='bold')
axes[1].set_title('Ensemble Methods: In Top-10 (✓=Yes, ✗=No)', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('ensemble_comparison_improved.png', dpi=150, bbox_inches='tight')
print("\nPlot saved: ensemble_comparison_improved.png")
plt.show()

In [14]:
# Save detailed results to CSV
results_df = pd.DataFrame({
    'user': users,
    'AE_score': score_ae,
    'AESkip_score': score_skip,
    'VAE_score': score_vae,
    'VAESkip_score': score_vskip,
    'AE_rank': scores_ranks['AE'],
    'AESkip_rank': scores_ranks['AESkip'],
    'VAE_rank': scores_ranks['VAE'],
    'VAESkip_rank': scores_ranks['VAESkip'],
    'is_insider': [u in INSIDERS for u in users]
})

# Add best ensemble ranking
best_ens_name = best_ensemble[0]
best_ens_scores = {}
if best_ens_name == 'minmax_avg':
    best_ens_scores = ens_minmax_avg
elif best_ens_name == 'zscore_avg':
    best_ens_scores = ens_zscore_avg
elif best_ens_name == 'rrf':
    best_ens_scores = ens_rrf
elif best_ens_name == 'rrf_weighted':
    best_ens_scores = ens_rrf_weighted
elif best_ens_name == 'consensus':
    best_ens_scores = ens_consensus

results_df[f'{best_ens_name}_score'] = best_ens_scores
results_df[f'{best_ens_name}_rank'] = to_ranks(best_ens_scores)

results_df = results_df.sort_values(f'{best_ens_name}_rank')
results_df.to_csv('results_improved_ensemble.csv', index=False)
print("Results saved: results_improved_ensemble.csv")
print("\nTop 15 detected anomalies:")
print(results_df[['user', f'{best_ens_name}_rank', 'is_insider']].head(15).to_string(index=False))

In [15]:
# Final recommendations
print("\n" + "="*60)
print("RECOMMENDATIONS FOR TOP-10 INSIDER DETECTION")
print("="*60)
print(f"""
1. **Model Selection**:
   - Use 2-3 models with best individual Top-10 performance
   - Avoid weak models that inflate ensemble scores

2. **Regularization Benefits**:
   - Dropout (30%) helps prevent overfitting
   - Reduced model capacity (latent dim: 6 vs 8) improves generalization
   - Early stopping on validation loss prevents train/val divergence

3. **Ensemble Strategy**:
   - Use RRF (Reciprocal Rank Fusion) for robustness
   - Weight models by their Top-10 performance
   - Consider consensus voting for conservative predictions

4. **Operational Deployment**:
   - Flag users in Top-10 for immediate investigation
   - Use model confidence scores for priority ranking
   - Monitor model performance on new data regularly

5. **Future Improvements**:
   - Collect more labeled insider threat data
   - Implement temporal features (trend analysis)
   - Add external threat intelligence integration
""")
print("="*60)